In [1]:
'''
Bibliotecas usadas
'''

import os
from datetime import datetime
import re
import subprocess
import warnings
import numpy as np
import pandas as pd
from collections import Counter
from Bio import Entrez
from Bio import SeqIO
from Bio.Seq import Seq
from sklearn.model_selection import (
    GroupShuffleSplit, 
    StratifiedGroupKFold, 
    RandomizedSearchCV, 
    GroupKFold
    )
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    make_scorer,
    matthews_corrcoef,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import subprocess
import shap
import seaborn as sns
from sklearn.inspection import permutation_importance


warnings.filterwarnings("ignore")

print("Bibliotecas importadas")


Bibliotecas importadas


In [2]:
'''
obtenção dos datasets
'''

EMAIL = "marcela.leite@gmail.com.br"
OUTPUT_DIR = "pipelineAG"
os.makedirs(OUTPUT_DIR, exist_ok=True)
Entrez.email = EMAIL

# ==========================================================
# QUERIES
# ==========================================================

POSITIVE_QUERY = r'''
    ("Solanaceae"[Organism] NOT ("Solanum betaceum"[Organism] OR "Solanum melongena"[Organism] OR "Solanum tuberosum"[Organism] OR "Physalis peruviana"[Organism])) 
    AND (TMV OR ToMV OR ToBRFV OR PMMoV OR TMGMV OR PVY OR PepYMV OR TEV OR PVA OR PVX OR TSWV OR GRSV OR TCSV OR CMV OR ToCV OR TICV OR AMV    
    OR "Rx1" OR "Sw-5" OR "Tm-2" OR "Tm-22" OR "RCY1" OR "Rx" OR "N GENE"
    OR "NLR" OR "NBS-LRR" OR "NB-LRR" OR "TIR-NBS-LRR" OR "CC-NBS-LRR" OR "TNL" OR "CNL Domain" OR "CNL"
    OR "NBS-ARC" OR "NB-ARC domain" OR "NB-ARC" OR "Nucleotide-binding site"
    OR "RDR" OR "Dicer-like" OR "Argonaute" OR "AGO" OR "eIF4E" )
    NOT (partial OR fragment OR hypothetical OR predicted OR DAP-seq OR "transcription factor" OR DNA-binding OR microtom)    
    NOT ("pathogenesis-reletad protein" OR osmotin OR PR1 OR PR2 OR PR3 OR PR4 OR PR5 OR synthase OR metabolic OR precursor)
    NOT txid10239[Organism:exp]
'''
NEGATIVE_QUERY = r'''
    ("Solanaceae"[Organism] NOT ("Solanum melongena"[Organism] OR "Solanum tuberosum"[Organism] OR "Physalis peruviana"[Organism])) 
   AND 100:3000[Sequence Length] 
   NOT (TMV OR ToMV OR ToBRFV OR PMMoV OR TMGMV OR PVY OR PepYMV OR TEV OR PVA OR PVX OR TSWV OR GRSV OR TCSV OR CMV OR ToCV OR TICV OR AMV    
    OR "Rx1" OR "Sw-5" OR "Tm-2" OR "Tm-22" OR "RCY1" OR "Rx" OR "N"
    OR "virus resistance" OR "antiviral" OR "Tobacco mosaic virus" OR "Potato virus" 
    OR "NLR" OR "NBS-LRR" OR "NB-LRR" OR "TIR-NBS-LRR" OR "CC-NBS-LRR" OR "TNL" OR "CNL Domain" OR "CNL"
    OR "NBS-ARC" OR "NB-ARC domain" OR "NB-ARC" OR "Nucleotide-binding site"
    OR "RNA silencing" OR "RDR" OR "Dicer-like" OR "Argonaute" OR "AGO" OR "eIF4E" )
   NOT (partial OR fragment OR hypothetical OR predicted OR uncharacterized OR DAP-seq OR "transcription factor" OR DNA-binding OR microtom )
'''

ARABIDOPSIS_QUERY = '''
   "Arabidopsis thaliana"[Organism] AND (biomol_mrna[PROP] OR tsa[keyword])
      AND 1500:30000[Sequence Length]
'''

SOLANUM_MELONGENA = '''
    "Solanum melongena"[Organism] AND (biomol_mrna[PROP] OR tsa[keyword])
      AND 1500:30000[Sequence Length]
    '''

TUBEROSUM_QUERY = '''
    "Solanum tuberosum"[Organism] AND (biomol_mrna[PROP] OR tsa[keyword])
      AND 1500:30000[Sequence Length]
    '''

PHYSALIS_QUERY = '''
  "Physalis peruviana"[Organism] AND (biomol_mrna[PROP] OR tsa[keyword])
      AND 1500:30000[Sequence Length]
      '''

# ==========================================================
# DOWNLOAD NCBI
# ==========================================================

def baixar_proteinas_ncbi(query, output_fasta, max_records=35000):
    print("\n================================")
    print("DOWNLOAD NCBI: ",output_fasta)
    print("================================")
    handle = Entrez.esearch(
        db="protein",
        term=query,
        retmax=max_records
    )
    record = Entrez.read(handle)
    ids = record["IdList"]
    print(f"Proteínas encontradas: {len(ids)}")
    if len(ids) == 0:
        return
    handle = Entrez.efetch(
        db="protein",
        id=ids,
        rettype="fasta",
        retmode="text"
    )
    with open(output_fasta, "w") as f:
        f.write(handle.read())
    print(f"Arquivo salvo: {output_fasta}")

def baixar_transcriptoma(output_fasta, query):
    print("\n================================")
    print("DOWNLOAD TRANSCRIPTOMA: ",output_fasta)
    print("================================")
    handle = Entrez.esearch(
        db="nucleotide",
        term=query,
        retmax=50000
    )
    record = Entrez.read(handle)
    ids = record["IdList"]
    print(f"Transcritos encontrados: {len(ids)}")
    handle = Entrez.efetch(
        db="nucleotide",
        id=ids,
        rettype="fasta",
        retmode="text"
    )
    with open(output_fasta, "w") as f:
        f.write(handle.read())
    print(f"Arquivo salvo: {output_fasta}")

#Dados de Entrada - baixar arquivos FASTA das sequências de proteínas
POSITIVE_FASTA = f"{OUTPUT_DIR}/positive.fasta"
NEGATIVE_FASTA = f"{OUTPUT_DIR}/negative.fasta"


print("\n============ FINALIZADO CONFIGURAÇÃO DE PARÂMETROS ============\n")



============ FINALIZADO CONFIGURAÇÃO DE PARÂMETROS ============



In [3]:
inicio = datetime.now()
print("\n\nInciando download da classe positiva: ", inicio.strftime("%H:%M:%S"))
baixar_proteinas_ncbi(POSITIVE_QUERY,POSITIVE_FASTA)
fim = datetime.now()
print("\n\nFinalizado em:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)



Inciando download da classe positiva:  10:32:10

DOWNLOAD NCBI:  pipelineAG/positive.fasta
Proteínas encontradas: 8914
Arquivo salvo: pipelineAG/positive.fasta


Finalizado em: 10:33:03

Tempo decorrido: 0:00:53.013381


In [4]:
inicio = datetime.now()
print("\n\nInciando download da classe negativa: ", inicio.strftime("%H:%M:%S"))
baixar_proteinas_ncbi(NEGATIVE_QUERY,NEGATIVE_FASTA)
fim = datetime.now()
print("\n\nFinalizado em:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)

print("\n\nArquivos salvos na pasta: ",OUTPUT_DIR)
print("============ FIM COLETA DE DADOS TREINO/VALIDAÇÂO ============")



Inciando download da classe negativa:  10:33:03

DOWNLOAD NCBI:  pipelineAG/negative.fasta
Proteínas encontradas: 35000
Arquivo salvo: pipelineAG/negative.fasta


Finalizado em: 10:34:03

Tempo decorrido: 0:01:00.093673


Arquivos salvos na pasta:  pipelineAG
============ FIM COLETA DE DADOS TREINO/VALIDAÇÂO ============


In [5]:
#Dados para aplicação - baixar transcriptomas - RNA-Seq
transcriptoma_arabidopsis_fasta = (f"{OUTPUT_DIR}/arabidopsis_transcriptoma.fasta")
transcriptoma_melongena_fasta = (f"{OUTPUT_DIR}/melongena_transcriptoma.fasta")
transcriptoma_tuberosum_fasta = (f"{OUTPUT_DIR}/tuberosum_transcriptoma.fasta")
transcriptoma_physalis_fasta = (f"{OUTPUT_DIR}/physalis_transcriptoma.fasta")

In [6]:
print("\n\nInciando downaload de dados de transcriptmas")
inicio = datetime.now()
print("\n\nInciando download ARABIDOPSIS: ", inicio.strftime("%H:%M:%S"))
baixar_transcriptoma(transcriptoma_arabidopsis_fasta, ARABIDOPSIS_QUERY)
fim = datetime.now()
print("\n\nFinalizado em:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)



Inciando downaload de dados de transcriptmas


Inciando download ARABIDOPSIS:  10:34:03

DOWNLOAD TRANSCRIPTOMA:  pipelineAG/arabidopsis_transcriptoma.fasta
Transcritos encontrados: 49951
Arquivo salvo: pipelineAG/arabidopsis_transcriptoma.fasta


Finalizado em: 10:34:46

Tempo decorrido: 0:00:43.022369


In [8]:
inicio = datetime.now()
print("\n\nInciando download PHYSALIS: ", inicio.strftime("%H:%M:%S"))
baixar_transcriptoma(transcriptoma_physalis_fasta, PHYSALIS_QUERY)
fim = datetime.now()
print("\n\nFinalizado em:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)



Inciando download PHYSALIS:  10:35:39

DOWNLOAD TRANSCRIPTOMA:  pipelineAG/physalis_transcriptoma.fasta
Transcritos encontrados: 21702
Arquivo salvo: pipelineAG/physalis_transcriptoma.fasta


Finalizado em: 10:36:21

Tempo decorrido: 0:00:42.376873


In [9]:
inicio = datetime.now()
print("\n\nInciando download MELONGENA: ", inicio.strftime("%H:%M:%S"))
baixar_transcriptoma(transcriptoma_melongena_fasta, SOLANUM_MELONGENA)
fim = datetime.now()
print("\n\nFinalizado em:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)



Inciando download MELONGENA:  10:36:21

DOWNLOAD TRANSCRIPTOMA:  pipelineAG/melongena_transcriptoma.fasta
Transcritos encontrados: 8555
Arquivo salvo: pipelineAG/melongena_transcriptoma.fasta


Finalizado em: 10:36:59

Tempo decorrido: 0:00:38.041127


In [10]:
inicio = datetime.now()
print("\n\nInciando download TUBEROSUM: ", inicio.strftime("%H:%M:%S"))
baixar_transcriptoma(transcriptoma_tuberosum_fasta, TUBEROSUM_QUERY)
fim = datetime.now()
print("\n\nFinalizado em:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)

print("\n\n\nArquivos salvos na pasta: ",OUTPUT_DIR)
print("============ FIM COLETA DE DADOS APLICAÇÃO ============")



Inciando download TUBEROSUM:  10:36:59

DOWNLOAD TRANSCRIPTOMA:  pipelineAG/tuberosum_transcriptoma.fasta
Transcritos encontrados: 20832
Arquivo salvo: pipelineAG/tuberosum_transcriptoma.fasta


Finalizado em: 10:37:42

Tempo decorrido: 0:00:43.120825



Arquivos salvos na pasta:  pipelineAG
============ FIM COLETA DE DADOS APLICAÇÃO ============


In [11]:
'''
Tratamento dos dados
    - Criar Dataframes com os conjuntos de dados de treino e validação
    - Executar CD-Hit para eliminar sequencias muitos semelhantes
    - Extração de atributos (features)

'''

# ==========================================================
# AAC
# ==========================================================
# composição de aminoácidos - transformar a proteína em dados numéricos
# conta a frequência de cada aminoácido em uma sequẽncia de proteína
# como são 20 aminoácidos possíveis, irá gerar 20 features com seus percentuais
# com essa frequência consegue ter um parâmetro para distinguir as proteínas
# por exemplo proteínas de resistência tem geralmente certa frequência de aminoácidos
# mas ignora a ordem - por isso a sequẽncia de dipeptídeos irá incrementar o modelo considerando a ordem
# sequência de 20 aminoácidos
AMINO_ACIDOS = list("ACDEFGHIKLMNPQRSTVWY")

motifs = {
    # ==========================
    # NLR (CNL/TNL)
    # ==========================
    "P_LOOP": r"[AGST].{3,5}GK[TS]",
    "KINASE2": r"[ALIVMFYW]{3,5}DD[VILW][WFY]",
    "GLPL": r"G[LVI][PAST][LIVAF]",
    "MHD": r"[MVIL][HN][DE][VILAST]",
    "RNBS_A": r"F.DL.{0,3}AW",
    "RNBS_B": r"G[SA].{2,6}T",
    "RNBS_C": r"Y.[VIL].{1,3}[LS]",
    "RNBS_D": r"C.{2,8}P",

    # ==========================
    # Protein Kinase (KIN/RLK)
    # ==========================
    "HRD": r"HRD[LIVMFY]",
    "DFG": r"DFG",

    # ==========================
    # LRR (RLP/RLK)
    # ==========================
    "LRR1": r"L..L..L..N",
    "LRR2": r"L[A-Z]{2}L[A-Z]{2}L[A-Z]{2}[NCT]"

}

def calcular_aac(seq):
    seq = seq.upper()
    total = len(seq)
    if total == 0:
        return [0] * len(AMINO_ACIDOS)
    counts = Counter(seq)
    return [
        counts.get(aa, 0) / total
        for aa in AMINO_ACIDOS
    ]

# ==========================================================
# DIPEPTÍDEOS
# ==========================================================
'''
encontrar pares de aminoácidos consectíveis em uma sequência protéica  -sequência de dois aminoácidos
considera a ordem dos aminoácidos
há certos padrões comuns para 
para 20 aminoácidos, há possibilidade de 20*20 = 400 dipeptídeos - features
'''

DIPEPTIDES = [
    a + b
    for a in AMINO_ACIDOS
    for b in AMINO_ACIDOS
]

def calcular_dipeptideos(seq):
    seq = seq.upper()
    total = len(seq) - 1
    if total <= 0:
        return [0] * len(DIPEPTIDES)
    counts = Counter([
        seq[i:i+2]
        for i in range(total)
    ])
    return [
        counts.get(dp, 0) / total
        for dp in DIPEPTIDES
    ]

def localizar_motivos(seq):
    posicoes = {}
    for nome, pattern in motifs.items():
        match = re.search(pattern, seq)

        if match:
            posicoes[nome] = match.start()
        else:
            posicoes[nome] = None

    return posicoes
    

def calcular_distancias(posicoes):

    def distancia(a, b):
        if posicoes[a] is None or posicoes[b] is None:
            return -1
        return posicoes[b] - posicoes[a]

    return {
        "dist_ploop_kinase2": distancia("P_LOOP", "KINASE2"),
        "dist_kinase2_glpl": distancia("KINASE2", "GLPL"),
        "dist_glpl_mhd": distancia("GLPL", "MHD")
    }
    
def contar_motivos(posicoes):
    return sum(
        1 for v in posicoes.values()
        if v is not None
    )
    
def extrair_motifs(seq):

    return [
        int(bool(re.search(pattern, seq)))
        for pattern in motifs.values()
    ]



# ==========================================================
# FEATURES - atributos
# ==========================================================
#calculando features
def extrair_features_proteina(seq):
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)
    if len(seq) < 50:
        return None
    features = []
    features.append(len(seq)) # 1 - comprimento da cadeia
    features.extend(calcular_aac(seq)) # 20 composição de aminoácidos - frequência de cada um na sequência
    features.extend(calcular_dipeptideos(seq)) #400 possibilidades - considera a ordem dos aminoácidos para buscar os mais frequêntes em proteínas de resistência
    
    features.extend(extrair_motifs(seq))
    # posições dos motivos
    posicoes = localizar_motivos(seq)
    # número de motivos encontrados
    num_motifs_detectados = contar_motivos(posicoes)
    # distâncias entre motivos
    distancias = calcular_distancias(posicoes)
    
    features.append(num_motifs_detectados)
    protein_length = len(seq)

    features.append(distancias["dist_ploop_kinase2"])
    features.append(distancias["dist_kinase2_glpl"])
    features.append(distancias["dist_glpl_mhd"])
    
    features.append(
        distancias["dist_ploop_kinase2"]/protein_length
        if distancias["dist_ploop_kinase2"] != -1 else -1
    )
    features.append(
        distancias["dist_kinase2_glpl"]/protein_length
        if distancias["dist_kinase2_glpl"] != -1 else -1
    )
    features.append(
        distancias["dist_glpl_mhd"]/protein_length
        if distancias["dist_glpl_mhd"] != -1 else -1
    )
        
    features.append(int(posicoes["P_LOOP"] is not None))
    features.append(int(posicoes["KINASE2"] is not None))
    features.append(int(posicoes["GLPL"] is not None))
    features.append(int(posicoes["MHD"] is not None))

    encontrados = {}    
    for nome, regex in motifs.items():
        encontrados[nome] = int( re.search(regex, seq) is not None)

    # features.append(encontrados[nome])
    
    num_nlr = (
        encontrados["P_LOOP"] +
        encontrados["KINASE2"] +
        encontrados["GLPL"] +
        encontrados["MHD"]
    )

    # num_kinase = (
    #     encontrados["HRD"] +
    #     encontrados["DFG"]
    # )

    # num_lrr = (
    #     encontrados["LRR1"] +
    #     encontrados["LRR2"]
    # )

    features.extend([
        num_nlr,
        # num_kinase,
        # num_lrr
    ])
    
    return features
    
# ==========================================================
# DATASET
# ==========================================================
motif_cols = [
    "P_LOOP",
    "KINASE2",
    "GLPL",
    "MHD",
    "RNBS_A",
    "RNBS_B",
    "RNBS_C",
    "RNBS_D",
    "HRD",
    "DFG",
    "LRR1",
    "LRR2",
    "NUM_NLR_MOTIFS"
 ]
    # "NUM_KINASE_MOTIFS",
    # "NUM_LRR_MOTIFS"

def criar_dataset(positive_fasta, negative_fasta):
    data = []
    columns = (
        ["protein_length"]
        + [f"AAC_{aa}" for aa in AMINO_ACIDOS]
        + [f"DIPEP_{dp}" for dp in DIPEPTIDES]
        + motif_cols
        + ["num_motifs_detectados"]
        + ["dist_ploop_kinase2"]
        + ["dist_kinase2_glpl"]
        + ["dist_glpl_mhd"]
        + ["dist_ploop_kinase2_norm"]
        + ["dist_kinase2_glpl_norm"]
        + ["dist_glpl_mhd_norm"]        
        + ["has_ploop"]
        + ["has_kinase2"]
        + ["has_glpl"]
        + ["has_mhd"]
    )

    # POSITIVOS
    for record in SeqIO.parse(positive_fasta, "fasta"):
        seq = str(record.seq)
        feats = extrair_features_proteina(seq)
        if feats is None:
            continue
        row = feats + [1, record.id]
        data.append(row)

    # NEGATIVOS
    for record in SeqIO.parse(negative_fasta, "fasta"):
        seq = str(record.seq)
        feats = extrair_features_proteina(seq)
        if feats is None:
            continue
        row = feats + [0, record.id]
        data.append(row)
        
    df = pd.DataFrame(
        data,
        columns=columns + ["target", "id"]
    )
    print("\nTotal de Features/Atributos:",len(columns)) # quantidade
    print("\nFeatures/atributos:")
    print(columns)     
    return df, columns

# CD-Hit
# chama o programa CD-HIT do sistema operacional
def executar_cdhit(
    input_fasta,
    output_fasta,
    identity=0.7 
):
    cmd = [
        "cd-hit",
        "-i", input_fasta,
        "-o", output_fasta,
        "-c", str(identity),
        "-n", "5",
        "-M", "30000",
        "-T", "4"
    ]

    print("\nExecutando CD-HIT...")
    subprocess.run(cmd, check=True)
    print("CD-HIT concluído.")

def limpar(seq_id):
    seq_id = seq_id.strip()
    seq_id = seq_id.split()[0]
    return seq_id

def ler_clusters_cdhit(cluster_file):
    clusters = {}
    cluster_id = None
    with open(cluster_file) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">Cluster"):
                cluster_id = line.replace(">Cluster ", "")
                clusters[cluster_id] = []
            else:
                seq_id = line.split(">")[1].split("...")[0]
                seq_id = limpar(seq_id)
                clusters[cluster_id].append(seq_id)
    return clusters

inicio = datetime.now()
print("\n\n\nInciando Tratamento dos dados: ", inicio.strftime("%H:%M:%S"))

# executa o CD-Hit nos arquivos baixados
print("\n\nInciar a execução do CD-HIT\n")
executar_cdhit(
    POSITIVE_FASTA,
    f"{OUTPUT_DIR}/positive_nr.fasta",
    identity=0.7
)

executar_cdhit(
    NEGATIVE_FASTA,
    f"{OUTPUT_DIR}/negative_nr.fasta",
    identity=0.7
)
# ======================================================
# Montar os DATASETs
# ======================================================

POSITIVE_FASTA = f"{OUTPUT_DIR}/positive_nr.fasta"
NEGATIVE_FASTA = f"{OUTPUT_DIR}/negative_nr.fasta"


df, columns = criar_dataset(
    POSITIVE_FASTA,
    NEGATIVE_FASTA
)

df["id"] = df["id"].apply(limpar)
# ======================================================
# CLUSTERS
# ======================================================

clusters_pos = ler_clusters_cdhit(
    f"{OUTPUT_DIR}/positive_nr.fasta.clstr"
)

clusters_neg = ler_clusters_cdhit(
    f"{OUTPUT_DIR}/negative_nr.fasta.clstr"
)

cluster_map = {}

for c, seqs in clusters_pos.items():
    for s in seqs:
        cluster_map[s] = f"POS_{c}"

for c, seqs in clusters_neg.items():
    for s in seqs:
        cluster_map[s] = f"NEG_{c}"

df["cluster"] = df["id"].map(cluster_map)

print("\n Clusters ausentes: ")
print(df["cluster"].isna().sum())
df = df.dropna(subset=["cluster"])

print("\nDataset original pós CD-HIT:", df.shape)

# ======================================================
# NOVO BLOCO: BALANCEAMENTO IMPERATIVO (DOWNSAMPLING)
# ======================================================
print("\n--- Executando Balanceamento de Classes ---")
df_positivos = df[df["target"] == 1]
df_negativos = df[df["target"] == 0]

print(f"Contagem inicial -> Positivos (R-genes): {len(df_positivos)} | Negativos (Background): {len(df_negativos)}")

# Se houver mais negativos do que positivos (cenário padrão), reduzimos os negativos
if len(df_negativos) > len(df_positivos):
    # O sample() escolhe aleatoriamente N linhas do grupo majoritário
    df_negativos_balanceados = df_negativos.sample(n=len(df_positivos), random_state=42)
    df = pd.concat([df_positivos, df_negativos_balanceados]).reset_index(drop=True)
    print(f"Subamostragem aplicada com sucesso nos Negativos.")
else:
    print("O dataset já está balanceado ou possui mais positivos que negativos (incomum). Nenhuma ação foi tomada.")

print(f"Contagem final   -> Positivos: {len(df[df['target'] == 1])} | Negativos: {len(df[df['target'] == 0])}")
print("Dataset Final Pronto:", df.shape)


fim = datetime.now()
print("\n\nFinalizado em:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)
print("\n\nFim tratamento dos dados...")






Inciando Tratamento dos dados:  10:37:42


Inciar a execução do CD-HIT


Executando CD-HIT...
Program: CD-HIT, V4.8.1 (+OpenMP), Aug 20 2021, 08:39:56
Command: cd-hit -i pipelineAG/positive.fasta -o
         pipelineAG/positive_nr.fasta -c 0.7 -n 5 -M 30000 -T 4

Started: Tue Jun 16 10:37:42 2026
                            Output                              
----------------------------------------------------------------
total seq: 8914
longest and shortest : 3540 and 51
Total letters: 7801441
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 8M
Buffer          : 4 X 11M = 45M
Table           : 2 X 65M = 130M
Miscellaneous   : 0M
Total           : 185M

Table limit with the given memory limit:
Max number of representatives: 4000000
Max number of word counting entries: 3726820522

# comparing sequences from          0  to       1485
.---------- new table with      236 representatives
# comparing sequences from       1485  to       2723
99.9%---

In [12]:
'''
k-Fold-Cross-Validation

'''

def validacao_cruzada(nome, model, X_train, y_train, groups_train):
    
    cv = StratifiedGroupKFold(
        n_splits=5, #10
        shuffle=True,
        random_state=42
    )
    mcc_scorer = make_scorer(matthews_corrcoef)
    scoring_dict = {
        'recall': 'recall',
        'roc_auc': 'roc_auc',
        'mcc': mcc_scorer,
        'accuracy': 'accuracy',
        'precision': 'precision',
        'f1': 'f1'
    }
    scores = cross_validate(
        model,
        X_train,
        y_train,
        groups=groups_train,
        cv=cv,
        scoring=scoring_dict,
        n_jobs=-1
    )
    print(f"Finalizada a validação cruzada para: {nome}")
    return scores  
  

In [13]:
'''
Criação dos modelos/ Treinamento
'''
# para melhor separar os conjuntos de treino e teste considerando os agrupamentos feitos no CD-HIT para sequẽncias muito parecidas
def split_por_homologia(df):
    print(df.head(30))
    groups = df["cluster"]
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )
    # cv  = StratifiedGroupKFold(
    #     n_splits = 10,
    #     shuffle=True,
    #     random_state = 42
    # )    

    train_idx, test_idx = next(
        splitter.split(df, groups=groups)
        # cv.split(df, df["target"], groups)
    )
    train_df = df.iloc[train_idx]
    test_df = df.iloc[test_idx]
    return train_df, test_df

def calcular_e_salvar_shap(nome, model, X_test, feature_names, output_dir):
    """
    Calcula os SHAP values para modelos de árvore e salva os gráficos explicativos.
    """
    print(f"\n[SHAP] Inicializando análise de interpretabilidade para: {nome}...")
    
    # 1. Inicializa o explicador otimizado para árvores
    explainer = shap.TreeExplainer(model)
    
    # 2. Calcula os SHAP values para o conjunto de teste
    shap_values = explainer.shap_values(X_test)
    
    # 3. Tratamento de formato (Scikit-Learn Random Forest vs XGBoost)
    # O Random Forest do sklearn retorna uma lista contendo [valores_classe_0, valores_classe_1]
    # O XGBoost retorna diretamente a matriz de SHAP values para a classe alvo
    if isinstance(shap_values, list):
        # Selecionamos o índice 1 (classe positiva: proteínas de resistência/alvos)
        shap_values_pos = shap_values[1]
    elif isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
        # Tratamento para versões específicas do SHAP que geram arrays 3D
        shap_values_pos = shap_values[:, :, 1]
    else:
        shap_values_pos = shap_values

    # 4. Gráfico 1: Summary Plot (Beeswarm)
    # Mostra o impacto biológico de cada feature (ex: se alta frequência aumenta ou diminui a predição)
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values_pos, 
        X_test, 
        feature_names=feature_names, 
        max_display=20,  # Foco nas top 20 características
        show=False
    )
    # plt.title(f"SHAP Summary Plot (Top 20) - {nome}", fontsize=14, pad=20)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{nome}_shap_summary.png", bbox_inches="tight", dpi=300)
    plt.close()
    
    # 5. Gráfico 2: Bar Plot (Importância Global)
    # Mostra a magnitude média do impacto de cada feature de forma absoluta
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values_pos, 
        X_test, 
        feature_names=feature_names, 
        plot_type="bar", 
        max_display=20, 
        show=False
    )
    # plt.title(f"SHAP Global Feature Importance - {nome}", fontsize=14, pad=20)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{nome}_shap_bar.png", bbox_inches="tight", dpi=300)
    plt.close()
    
    print(f"[SHAP] Gráficos salvos com sucesso em '{output_dir}/'!")


def calcular_shap_svm(nome, model, feature_names,  X_train, X_test, y_test, output_dir):
    # ======================================================
    # IMPORTÂNCIA POR PERMUTAÇÃO PARA O SVM (Novo)
    # ======================================================
    print("\n[SVM] Calculando a importância por permutação dos atributos...")
    
    # Calcula a permutação no X_test (pode usar X_train se quiser focar no ajuste ao treino)
    result_svm = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
    
    # Monta o DataFrame com os resultados
    importancia_svm = pd.DataFrame({
        'feature': columns,
        'importance_mean': result_svm.importances_mean
    }).sort_values('importance_mean', ascending=False)
    
    print("\nTop 20 features mais importantes para o SVM:")
    print(importancia_svm.head(20))
    
    # Gera o gráfico de barras igual ao do XGBoost
    top_svm = importancia_svm.head(20)
    plt.figure(figsize=(10,8))
    plt.barh(top_svm["feature"][::-1], top_svm["importance_mean"][::-1], color='seagreen')
    plt.xlabel("Queda no Desempenho (Média)")
    plt.ylabel("Feature / Atributo")
    # plt.title("Top 20 Features - SVM Permutation Importance")
    plt.tight_layout()
    plt.savefig(f"{output_dir}/svm_importancia_permutacao.png", bbox_inches="tight", dpi=300)
    plt.close()

    idx_back = np.random.choice(
        len(X_train),
        min(100,len(X_train)),
        replace = False
    )
    back = X_train[idx_back]
    
    idx_test = np.random.choice(
        len(X_test),
        min(200,len(X_test)),
        replace = False
    )
    x_test_sample = X_test[idx_test]
    
    print('\nCriando explainer shap...\n')
    explainer = shap.Explainer(model.predict, back)
    shap_values = explainer(x_test_sample, max_evals=1000)
    print("\nShap SHAP:",shap_values.values.shape)
    
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values.values, 
        x_test_sample, 
        feature_names=feature_names, 
      #  plot_type="bar", 
        max_display=20, 
        show=False
    )
    # plt.title(f"SHAP Global Feature Importance - {nome}", fontsize=14, pad=20)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{nome}_shap_bar_summary.png", bbox_inches="tight", dpi=300)
    plt.close()
    
    print(f"[SVM] Gráfico salvo com sucesso em '{output_dir}/svm_importancia_permutacao.png'!")
    
# ==========================================================
# TREINAMENTO - treina os modelos
# ==========================================================

def avaliar_modelo(nome, model, X_train, y_train, X_test, y_test):
    print("\n============================================================")
    print(nome)
    print("============================================================")
    model.fit(X_train, y_train)
    
    probs = model.predict_proba(X_test)[:,1]
    preds = (probs >= 0.5).astype(int)
    print(classification_report(y_test, preds))
    roc = roc_auc_score(y_test, probs)
    pr = average_precision_score(y_test, probs)
    mcc = matthews_corrcoef(y_test, preds)
    f1 = f1_score(y_test, preds)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    bal_acc = balanced_accuracy_score(y_test, preds)
    acc = accuracy_score(y_test, preds)

    print("ROC-AUC          :", roc)
    print("PR-AUC           :", pr)
    print("MCC              :", mcc)
    print("F1               :", f1)
    print("Precision        :", precision)
    print("Recall           :", recall)
    print("Balanced Accuracy:", bal_acc)
    print("Standard Accuracy:", acc)

    # ======================================================
    # MATRIZ DE CONFUSÃO
    # ======================================================
    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(5,5))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Negativo", "Positivo"]
    )
    disp.plot(cmap="Blues")
    # plt.title(f"Matriz de Confusão - {nome}")
    plt.savefig(
        f"{OUTPUT_DIR}/{nome}_matriz_confusao.png",
        bbox_inches="tight"
    )
    plt.close()
    # ======================================================
    # CURVA ROC
    # ======================================================
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6,6))
    plt.plot(
        fpr,
        tpr,
        label=f"AUC = {roc_auc:.4f}"
    )
    plt.plot([0,1], [0,1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    # plt.title(f"Curva ROC - {nome}")
    plt.legend(loc="lower right")
    plt.savefig(
        f"{OUTPUT_DIR}/{nome}_roc.png",
        bbox_inches="tight"
    )
    plt.close()
    
    return {
        "Modelo": nome,
        "ROC_AUC": roc,
        "F1": f1,
        "Precision": precision,
        "Recall": recall,
        "Balanced_Accuracy": bal_acc,
        "Accuracy": acc,
        "MCC":mcc
    }, model


# ======================================================
# Separação Treino/Teste 80/20
# SPLIT POR HOMOLOGIA
# ======================================================
inicio_treinamento = datetime.now()
print("\n\n\nInciando Treinamento dos modelos: ", inicio_treinamento.strftime("%H:%M:%S"))

train_df, test_df = split_por_homologia(df)
X_train = train_df[columns]
y_train = train_df["target"]
X_test = test_df[columns]
y_test = test_df["target"]
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ======================================================
# MODELOS
# ======================================================

rf = RandomForestClassifier(
    n_estimators=1000,
    # max_depth = 12,
    max_depth = None,
    min_samples_split=5,
    max_features=0.2,
    random_state=42,
    class_weight="balanced",
    n_jobs = 1
)    

xgb = XGBClassifier(
    n_estimators=1200,
    max_depth=7,
    learning_rate=0.05,
    # early_stopping_rounds=50,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    min_child_weight=3,
    reg_alpha=0.5,
    reg_lambda=2,
    scale_pos_weight=1.0
)


svm = SVC(
    probability=True,
    kernel="rbf",
    class_weight="balanced",
    random_state=42,
    C=2,
    gamma='scale'
)
scores = []
groups_train = train_df["cluster"]

# msc_scorer = make_scorer(matthews_corrcoef)

scoring = {
    'mcc': make_scorer(matthews_corrcoef),
    'recall':  make_scorer(recall_score),
    'precision':  make_scorer(precision_score),
    'f1':  make_scorer(f1_score),
    'roc_auc': 'roc_auc',
}





Inciando Treinamento dos modelos:  10:37:46
    protein_length     AAC_A     AAC_C     AAC_D     AAC_E     AAC_F  \
0              231  0.090909  0.012987  0.069264  0.077922  0.051948   
1              704  0.049716  0.007102  0.080966  0.117898  0.041193   
2              971  0.045314  0.017508  0.054583  0.071061  0.036045   
3              323  0.046440  0.027864  0.052632  0.080495  0.046440   
4              929  0.034446  0.034446  0.046286  0.069968  0.053821   
5              709  0.053597  0.014104  0.049365  0.071932  0.043724   
6              999  0.053053  0.018018  0.070070  0.050050  0.036036   
7             1071  0.056956  0.020542  0.059757  0.038282  0.043884   
8             1050  0.043810  0.022857  0.053333  0.080952  0.041905   
9              904  0.042035  0.022124  0.032080  0.055310  0.056416   
10            1152  0.065972  0.015625  0.041667  0.046875  0.030382   
11             949  0.033720  0.034773  0.052687  0.067439  0.041096   
12             27

In [14]:
inicio_otimizacao = datetime.now()
print("\n\n\nInciando Otimização dos modelos: ", inicio_otimizacao.strftime("%H:%M:%S"))

cv_t = GroupKFold(n_splits=5)

param_dist_xgb = {
    'n_estimators':[500,800,1200,1500],
    'max_depth':[3,4,5,6,7,10],
    'learning_rate':[0.01,0.03,0.05,0.1],
    'subsample':[0.7,0.8,0.9],
    'colsample_bytree':[0.7,0.8,0.9],
    'min_child_weight':[1,3,5],
    'reg_alpha':[0,0.1,0.5,0.7],
    'reg_lambda':[1,2,5]
}
# Otimização XGB
print('\n\nOtimizando XGBoost...')
search_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_xgb,
    n_iter=50,
    scoring=scoring,
    refit='f1',
    cv=cv_t,
    random_state=42,
    verbose=2,
    n_jobs=-1
)

search_xgb.fit(
    X_train,
    y_train,
    groups=groups_train
)
print("\nMelhores parâmetros para o XGB:")
print(search_xgb.best_params_)

print("\nMelhor Recall:")
print(search_xgb.best_score_)

best_params = search_xgb.best_params_
xgb = search_xgb.best_estimator_







Inciando Otimização dos modelos:  10:37:46


Otimizando XGBoost...
Fitting 5 folds for each of 50 candidates, totalling 250 fits

Melhores parâmetros para o XGB:
{'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 1500, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.7}

Melhor Recall:
0.8920809070696771


In [15]:
# otimização RF

param_dist_rf = {
    'n_estimators':[500,1000,1200,1500],
    'max_depth':[8,12,16, None],
    'min_samples_split':[2,4,5,8],
    'min_samples_leaf':[1,2,4],
    'max_features':['sqrt','log2',0,3],
    'class_weight':['balanced','balanced_subsample']
}

print('\n\nOtimizando RF...')
search_rf = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist_rf,
    n_iter=40,
    scoring=scoring,
    refit='f1',
    cv=cv_t,
    random_state=42,
    verbose=2,
    n_jobs=-1
)

search_rf.fit(
    X_train,
    y_train,
    groups=groups_train
)
print("\nMelhores parâmetros para o RF:")
print(search_rf.best_params_)

print("\nMelhor Recall:")
print(search_rf.best_score_)

rf = search_rf.best_estimator_

rf.set_params(n_jobs=-1)




Otimizando RF...
Fitting 5 folds for each of 40 candidates, totalling 200 fits

Melhores parâmetros para o RF:
{'n_estimators': 1000, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None, 'class_weight': 'balanced_subsample'}

Melhor Recall:
0.8609705821511874


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",1000
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",8
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metri

In [16]:
print('\n\nOtimizando SVM...')
param_dist_svm = {
    'C':[0.5,1,2,5,10,20,50],
    'gamma':[
        'scale',
        0.01,
        0.005,
        0.001,
        0.0005
    ],
    'kernel':['rbf']
}
search_svm = RandomizedSearchCV(
    estimator=svm,
    param_distributions=param_dist_svm,
    n_iter=25,
    scoring=scoring,
    refit='f1',
    cv=cv_t,
    random_state=42,
    verbose=2,
    n_jobs=-1
)

search_svm.fit(
    X_train,
    y_train,
    groups=groups_train
)
print("\nMelhores parâmetros para o SVM:")
print(search_svm.best_params_)

print("\nMelhor Recall:")
print(search_svm.best_score_)

svm = search_svm.best_estimator_

fim = datetime.now()
print("\n\nFinalizado otimização dos modelos em:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio_otimizacao)
# Fim otimização






Otimizando SVM...
Fitting 5 folds for each of 25 candidates, totalling 125 fits

Melhores parâmetros para o SVM:
{'kernel': 'rbf', 'gamma': 0.001, 'C': 2}

Melhor Recall:
0.8899564834649676


Finalizado otimização dos modelos em: 10:53:05

Tempo decorrido: 0:15:19.242567


In [17]:
# xgb = search_xgb.best_estimator_

print("\n\n Inciando validação cruzada");
scores_rf = validacao_cruzada('RF',rf,X_train, y_train, groups_train)
scores_xgb = validacao_cruzada('XGB',xgb, X_train, y_train, groups_train)
scores_svm = validacao_cruzada('SVM',svm, X_train, y_train, groups_train)

print("Cross-Validation:RF")
for k,v in scores_rf.items():
    if "test" in k:
        print(f"{k}: {v.mean():.4f} ± {v.std():.4f}")

print("Cross-Validation XGB")
for k,v in scores_xgb.items():
    if "test" in k:
        print(f"{k}: {v.mean():.4f} ± {v.std():.4f}")

print("Cross-Validation SVM")
for k,v in scores_svm.items():
    if "test" in k:
        print(f"{k}: {v.mean():.4f} ± {v.std():.4f}")


resultados_cv = pd.DataFrame({
    'Modelo': ['RF','XGB','SVM'],
    'Accuracy': [
        scores_rf['test_accuracy'].mean(),
        scores_xgb['test_accuracy'].mean(),
        scores_svm['test_accuracy'].mean(),
    ],    
    'Recall': [
        scores_rf['test_recall'].mean(),
        scores_xgb['test_recall'].mean(),
        scores_svm['test_recall'].mean(),
    ],
    'Precision': [
        scores_rf['test_precision'].mean(),
        scores_xgb['test_precision'].mean(),
        scores_svm['test_precision'].mean(),
    ],
    'F1': [
        scores_rf['test_f1'].mean(),
        scores_xgb['test_f1'].mean(),
        scores_svm['test_f1'].mean(),
    ],
    'ROC_AUC': [
        scores_rf['test_roc_auc'].mean(),
        scores_xgb['test_roc_auc'].mean(),
        scores_svm['test_roc_auc'].mean(),
    ],
    'MCC': [
        scores_rf['test_mcc'].mean(),
        scores_xgb['test_mcc'].mean(),
        scores_svm['test_mcc'].mean(),
    ]
})
print(resultados_cv)
resultados_cv.to_csv(
        f"{OUTPUT_DIR}/metricas_cv.csv",
        index=False
    )
dados = []

for score in scores_rf['test_roc_auc']:
    dados.append(['RF',score])
for score in scores_xgb['test_roc_auc']:
    dados.append(['XGB',score])
for score in scores_svm['test_roc_auc']:
    dados.append(['SVM',score])

grafico = pd.DataFrame(dados, columns=['Modelo','ROC_AUC'])
grafico.boxplot(by='Modelo')
plt.ylabel("ROC-AUC")
# plt.title("10-Fold Cross Validation")
plt.suptitle("")
plt.tight_layout()
# plt.show()
plt.savefig(
    f"{OUTPUT_DIR}/kfold_cross_validation_roc.png",
    bbox_inches="tight"
)
plt.close()







 Inciando validação cruzada
Finalizada a validação cruzada para: RF
Finalizada a validação cruzada para: XGB
Finalizada a validação cruzada para: SVM
Cross-Validation:RF
test_recall: 0.7692 ± 0.0139
test_roc_auc: 0.9397 ± 0.0082
test_mcc: 0.7708 ± 0.0146
test_accuracy: 0.8757 ± 0.0081
test_precision: 0.9815 ± 0.0052
test_f1: 0.8624 ± 0.0099
Cross-Validation XGB
test_recall: 0.8512 ± 0.0209
test_roc_auc: 0.9424 ± 0.0089
test_mcc: 0.8015 ± 0.0207
test_accuracy: 0.8985 ± 0.0107
test_precision: 0.9432 ± 0.0131
test_f1: 0.8946 ± 0.0119
Cross-Validation SVM
test_recall: 0.8440 ± 0.0205
test_roc_auc: 0.9484 ± 0.0085
test_mcc: 0.8070 ± 0.0204
test_accuracy: 0.9004 ± 0.0103
test_precision: 0.9546 ± 0.0189
test_f1: 0.8956 ± 0.0112
  Modelo  Accuracy    Recall  Precision        F1   ROC_AUC       MCC
0     RF  0.875735  0.769228   0.981456  0.862423  0.939706  0.770808
1    XGB  0.898529  0.851225   0.943151  0.894628  0.942398  0.801468
2    SVM  0.900368  0.843984   0.954608  0.895586  0.9484

In [18]:
################################################
# Treinamento
inicio = datetime.now()
print("\n\n\nInciando treinamento RF: ", inicio.strftime("%H:%M:%S"))
results = []
r1, rf_model = avaliar_modelo(
    "Random Forest",
    rf,
    X_train,
    y_train,
    X_test,
    y_test
) 
results.append(r1)
fim = datetime.now()
print("\n\nFinalizado treinamento RF em:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)

calcular_e_salvar_shap("Random Forest", rf_model, X_test, columns,OUTPUT_DIR )

inicio = datetime.now()
print("\n\n\nInciando treinamento XGBoost: ", inicio.strftime("%H:%M:%S"))
r2, xgb_model = avaliar_modelo(
    "XGBoost",
    xgb,
    X_train,
    y_train,
    X_test,
    y_test
)
results.append(r2)
fim = datetime.now()
print("\n\nFinalizado em XGBoost:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)

calcular_e_salvar_shap("XGBoost", xgb_model, X_test, columns,OUTPUT_DIR )

inicio = datetime.now()
print("\n\n\nInciando treinamento SVM: ", inicio.strftime("%H:%M:%S"))
r3, svm_model = avaliar_modelo(
    "SVM",
    svm,
    X_train,
    y_train,
    X_test,
    y_test
)
results.append(r3)
fim = datetime.now()
print("\n\nFinalizado em SVM:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)

calcular_shap_svm("SVM", svm_model,columns, X_train, X_test, y_test, OUTPUT_DIR)

fim = datetime.now()
print("\n\nFinalizado fase de treinamento em:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio_treinamento)

print("\nFim criação/treino dos modelos\n")





Inciando treinamento RF:  10:53:27

Random Forest
              precision    recall  f1-score   support

           0       0.85      0.98      0.91       358
           1       0.98      0.80      0.88       322

    accuracy                           0.90       680
   macro avg       0.91      0.89      0.90       680
weighted avg       0.91      0.90      0.90       680

ROC-AUC          : 0.9542749574933203
PR-AUC           : 0.9618177020542162
MCC              : 0.8037186718808346
F1               : 0.8805460750853242
Precision        : 0.9772727272727273
Recall           : 0.8012422360248447
Balanced Accuracy: 0.8922412297442659
Standard Accuracy: 0.8970588235294118


Finalizado treinamento RF em: 10:53:29

Tempo decorrido: 0:00:02.270760

[SHAP] Inicializando análise de interpretabilidade para: Random Forest...
[SHAP] Gráficos salvos com sucesso em 'pipelineAG/'!



Inciando treinamento XGBoost:  10:54:01

XGBoost
              precision    recall  f1-score   support

       

PermutationExplainer explainer:   8%|▊         | 17/200 [04:43<53:38, 17.59s/it]

[CV] END colsample_bytree=0.7, learning_rate=0.05, max_depth=3, min_child_weight=3, n_estimators=800, reg_alpha=0.5, reg_lambda=5, subsample=0.9; total time=  20.1s
[CV] END colsample_bytree=0.9, learning_rate=0.01, max_depth=4, min_child_weight=5, n_estimators=1500, reg_alpha=0.1, reg_lambda=1, subsample=0.9; total time= 1.1min
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, min_child_weight=1, n_estimators=500, reg_alpha=0, reg_lambda=1, subsample=0.9; total time=  43.6s
[CV] END colsample_bytree=0.9, learning_rate=0.01, max_depth=5, min_child_weight=5, n_estimators=1200, reg_alpha=0, reg_lambda=1, subsample=0.9; total time= 1.2min
[CV] END colsample_bytree=0.9, learning_rate=0.05, max_depth=7, min_child_weight=5, n_estimators=1200, reg_alpha=0.1, reg_lambda=2, subsample=0.7; total time=  52.0s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=4, min_child_weight=1, n_estimators=800, reg_alpha=0, reg_lambda=2, subsample=0.7; total time=  37.8s
[CV] END cols

PermutationExplainer explainer: 201it [54:08, 16.24s/it]                        



Shap SHAP: (200, 445)
[SVM] Gráfico salvo com sucesso em 'pipelineAG/svm_importancia_permutacao.png'!


Finalizado fase de treinamento em: 11:48:49

Tempo decorrido: 1:11:03.465000

Fim criação/treino dos modelos



<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

In [19]:
'''
Avaliação e Validação dos modelos
'''
print("\n\nInciando Validação dos modelos\n")
# verificar importância dos atributos/feature para o Random Forest
importancia = pd.DataFrame({"feature":columns,"importance":rf_model.feature_importances_})
importancia = importancia.sort_values("importance",ascending = False)
print("\nTop 20 features: Random Forest")
print(importancia.head(20))
# GRÁFICO
top = importancia.head(20)
plt.figure(figsize=(10,8))
plt.barh( top["feature"][::-1], top["importance"][::-1] )
plt.xlabel("Importância")
plt.ylabel("Feature")
# plt.title("Top 20 Features - Random Forest")
plt.tight_layout()
plt.savefig(
    f"{OUTPUT_DIR}/rf_importancia_variaveis.png",
    bbox_inches="tight"
)
plt.close()

# verificar importância dos atributos/feature para o XGBoost
importancia = pd.DataFrame({"feature":columns, "importance":xgb_model.feature_importances_})
importancia = importancia.sort_values("importance", ascending = False)
print("\nTop 20 features: XGBoost")
print(importancia.head(20))
# GRÁFICO
top = importancia.head(20)
plt.figure(figsize=(10,8))
plt.barh(top["feature"][::-1], top["importance"][::-1])
plt.xlabel("Importância")
plt.ylabel("Feature")
# plt.title("Top 20 Features - XGBoost")
plt.tight_layout()
plt.savefig(
    f"{OUTPUT_DIR}/xgb_importancia_variaveis.png",
    bbox_inches="tight"
)
plt.close()

print("\nCOMPARAÇÃO MODELOS")
print(pd.DataFrame(results))

results_df = pd.DataFrame(results)
results_df.to_csv(
        f"{OUTPUT_DIR}/metricas_comparacao.csv",
        index=False
    )

metrics = [
    "ROC_AUC",
    "F1",
    "Precision",
    "Recall",
    "Balanced_Accuracy"
]

for metric in metrics:
    plt.figure(figsize=(8,5))
    plt.bar(
        results_df["Modelo"],
        results_df[metric]
    )

    plt.ylim(0, 1)
    plt.ylabel(metric)
    # plt.title(f"Comparação dos Modelos - {metric}")
    for i, v in enumerate(results_df[metric]):
        plt.text(i, v + 0.01, f"{v:.3f}", ha='center')

    plt.savefig(
        f"{OUTPUT_DIR}/comparacao_{metric}.png",
        bbox_inches="tight"
    )
    plt.close()

modelos = {
    "Random Forest":rf_model,
    "XGBoost":xgb_model,
    "SVM":svm_model
    }
plt.figure(figsize=(7,7))
for nome, model in modelos.items():
    probs = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(
        fpr,
        tpr,
        label=f"{nome} AUC={roc_auc:.3f}"
    )
plt.plot([0,1],[0,1],'--')
plt.xlabel("FPR")
plt.ylabel("TPR")
# plt.title("Comparação ROC")
plt.legend()
plt.savefig(f"{OUTPUT_DIR}/roc_comparativa.png")
plt.close()

print("Fim avaliação / validação modelos ")




Inciando Validação dos modelos


Top 20 features: Random Forest
                     feature  importance
10                     AAC_L    0.038210
444                  has_mhd    0.035311
433           NUM_NLR_MOTIFS    0.034540
440       dist_glpl_mhd_norm    0.024666
1                      AAC_A    0.024544
209                 DIPEP_LK    0.023640
422                  KINASE2    0.020788
441                has_ploop    0.019688
204                 DIPEP_LE    0.019556
421                   P_LOOP    0.018196
439   dist_kinase2_glpl_norm    0.018104
0             protein_length    0.016066
250                 DIPEP_NL    0.015859
13                     AAC_P    0.014667
438  dist_ploop_kinase2_norm    0.012703
190                 DIPEP_KL    0.012615
437            dist_glpl_mhd    0.012052
150                 DIPEP_HL    0.011909
215                 DIPEP_LR    0.010945
424                      MHD    0.010195

Top 20 features: XGBoost
                     feature  importance
444   

In [20]:
'''
Funções para rodar a aplicação
'''

df_validacao = []

def validar_motifs(df,planta,modelo):
    print("\nVALIDAÇÃO BIOLÓGICA\n")
    counts = {}
    for motif_name, pattern in motifs.items():
        counts[motif_name] = 0
        for seq in df["protein_seq"]:
            if re.search(pattern, seq):
                counts[motif_name] += 1
    for k, v in counts.items():
        print(k, ":", v)

    counts['planta'] = planta
    counts['modelo'] = modelo
    
    df_validacao.append(counts)
    df_counts = pd.DataFrame([counts])
    df_counts.to_csv(
        f"{OUTPUT_DIR}/validacao_biologica_{modelo}_{planta}.csv",
        index=False
    )



def rodar_transdecoder(fasta_nt):
    cmd1 = [
        "TransDecoder.LongOrfs",
        "-t",
        fasta_nt,
        "--output_dir",
        f"{OUTPUT_DIR}"
    ]
    with open(fasta_nt+"_output1.txt", "a", encoding="utf-8") as log_file:
        subprocess.run(cmd1, 
                       stdout=log_file, 
                       stderr=subprocess.STDOUT,
                       text=True)

    cmd2 = [
        "TransDecoder.Predict",
        "-t",
        fasta_nt,
        "--output_dir",
        f"{OUTPUT_DIR}"
    ]
    with open(fasta_nt+"_output2.txt", "a", encoding="utf-8") as log_file:
        subprocess.run(cmd2, 
                       stdout=log_file, 
                       stderr=subprocess.STDOUT,
                       text=True)
    pep_file = fasta_nt + ".transdecoder.pep"
    return pep_file


def predizer_transcritos(fasta_nt,model,nmodel, output, planta):
    probabilidade = 0.85
    candidatos = []
    pep_file = rodar_transdecoder(fasta_nt)
    for record in SeqIO.parse(fasta_nt+".transdecoder.pep", "fasta"):
        protein = str(record.seq)
        feats = extrair_features_proteina(protein)
        if feats is None:
            continue
        X = scaler.transform([feats])
        prob = model.predict_proba(X)[0][1]
        candidatos.append({
            "id": record.id,
            "descricao": record.description,
            "protein_length": len(protein),
            "probabilidade_resistencia": prob,
            "protein_seq": protein
        })

    candidatos_df = pd.DataFrame(candidatos)
    candidatos_df = candidatos_df.sort_values(
        "probabilidade_resistencia",
        ascending=False
    )

    # top = candidatos_df.head(40)
    top = candidatos_df[candidatos_df["probabilidade_resistencia"] >= probabilidade]
    print("\nTOP CANDIDATOS com probabilidade de >= ")
    print(probabilidade)
    print(fasta_nt+", MODELO: "+nmodel)
    print(top[[
        "id",
        "descricao",
        "protein_length",
        "probabilidade_resistencia"
    ]])

    validar_motifs(top,planta,nmodel)

    candidatos_df.to_csv(
        f"{OUTPUT_DIR}/{output}",
        index=False
    )


In [21]:
'''
Aplicação - Modelo XGBoost
'''
print("\n\n=================Inciando Aplicação: Predição de Genes================\n\n")
print("\n\n Predição Modelo XGBoost")
inicio = datetime.now()
print("\n\n\nInciando predição ARABIDOPSIS: ", inicio.strftime("%H:%M:%S"))

print("\nPredição para ARABIDOPSIS\n")
predizer_transcritos(transcriptoma_arabidopsis_fasta, xgb_model,'XGBoost',output="predicoes_arabidopsis_xgboost.csv",planta="Arabidopsis thaliana")

fim = datetime.now()
print("\n\nFinalizado predição ARABIDOPSIS:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)




=================Inciando Aplicação: Predição de Genes================




 Predição Modelo XGBoost



Inciando predição ARABIDOPSIS:  11:48:50

Predição para ARABIDOPSIS


TOP CANDIDATOS com probabilidade de >= 
0.85
pipelineAG/arabidopsis_transcriptoma.fasta, MODELO: XGBoost
                     id                                          descricao  \
4437  NM_001344576.1.p1  NM_001344576.1.p1 GENE.NM_001344576.1~~NM_0013...   
4359  NM_001344486.1.p1  NM_001344486.1.p1 GENE.NM_001344486.1~~NM_0013...   
4358  NM_001344485.1.p1  NM_001344485.1.p1 GENE.NM_001344485.1~~NM_0013...   
4074  NM_001344104.1.p1  NM_001344104.1.p1 GENE.NM_001344104.1~~NM_0013...   
8895     NM_122936.5.p1  NM_122936.5.p1 GENE.NM_122936.5~~NM_122936.5.p...   
...                 ...                                                ...   
5172  NM_001345509.1.p1  NM_001345509.1.p1 GENE.NM_001345509.1~~NM_0013...   
5171  NM_001345508.1.p1  NM_001345508.1.p1 GENE.NM_001345508.1~~NM_0013...   
5243  NM_001345607

In [22]:
print("\nPredição para MELONGENA\n")
inicio = datetime.now()
print("\n\n\nInciando predição MELONGENA: ", inicio.strftime("%H:%M:%S"))
predizer_transcritos(transcriptoma_melongena_fasta, xgb_model,'XGBoost',output="predicoes_melongena_xgboost.csv",planta="Solanum melongena")
fim = datetime.now()
print("\n\nFinalizado predição MELONGENA:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)


Predição para MELONGENA




Inciando predição MELONGENA:  11:50:52

TOP CANDIDATOS com probabilidade de >= 
0.85
pipelineAG/melongena_transcriptoma.fasta, MODELO: XGBoost
                     id                                          descricao  \
3221  GBEF01023314.1.p1  GBEF01023314.1.p1 GENE.GBEF01023314.1~~GBEF010...   
3220  GBEF01023313.1.p1  GBEF01023313.1.p1 GENE.GBEF01023313.1~~GBEF010...   
4489  GBEF01026077.1.p1  GBEF01026077.1.p1 GENE.GBEF01026077.1~~GBEF010...   
4488  GBEF01026076.1.p1  GBEF01026076.1.p1 GENE.GBEF01026076.1~~GBEF010...   
9238  GBEF01044458.1.p1  GBEF01044458.1.p1 GENE.GBEF01044458.1~~GBEF010...   
...                 ...                                                ...   
5476  GBEF01028522.1.p1  GBEF01028522.1.p1 GENE.GBEF01028522.1~~GBEF010...   
5488  GBEF01028531.1.p1  GBEF01028531.1.p1 GENE.GBEF01028531.1~~GBEF010...   
5473  GBEF01028520.1.p1  GBEF01028520.1.p1 GENE.GBEF01028520.1~~GBEF010...   
4458  GBEF01026020.1.p1  GBEF01026020.1.p1 GENE.

In [23]:
print("\nPredição para TUBEROSUM\n")
inicio = datetime.now()
print("\n\n\nInciando predição TUBEROSUM: ", inicio.strftime("%H:%M:%S"))
predizer_transcritos(transcriptoma_tuberosum_fasta, xgb_model,'XGBoost',output="predicoes_tuberosum_xgboost.csv",planta="Solanum tuberosum")
fim = datetime.now()
print("\n\nFinalizado predição TUBEROSUM:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)


Predição para TUBEROSUM




Inciando predição TUBEROSUM:  11:52:38

TOP CANDIDATOS com probabilidade de >= 
0.85
pipelineAG/tuberosum_transcriptoma.fasta, MODELO: XGBoost
                      id                                          descricao  \
9242   XM_015312842.1.p1  XM_015312842.1.p1 GENE.XM_015312842.1~~XM_0153...   
7736   XM_015304639.1.p1  XM_015304639.1.p1 GENE.XM_015304639.1~~XM_0153...   
10090  XM_015314337.1.p1  XM_015314337.1.p1 GENE.XM_015314337.1~~XM_0153...   
7408   XM_015304012.1.p1  XM_015304012.1.p1 GENE.XM_015304012.1~~XM_0153...   
8916   XM_015312154.1.p1  XM_015312154.1.p1 GENE.XM_015312154.1~~XM_0153...   
...                  ...                                                ...   
7494   XM_015304165.1.p1  XM_015304165.1.p1 GENE.XM_015304165.1~~XM_0153...   
2441   XM_006357444.2.p1  XM_006357444.2.p1 GENE.XM_006357444.2~~XM_0063...   
10091  XM_015314339.1.p2  XM_015314339.1.p2 GENE.XM_015314339.1~~XM_0153...   
8323   XM_015310952.1.p1  XM_015310952

In [24]:
print("\nPredição para PHYSALIS\n")
inicio = datetime.now()
print("\n\n\nInciando predição PHYSALIS: ", inicio.strftime("%H:%M:%S"))
predizer_transcritos(transcriptoma_physalis_fasta, xgb_model,'XGBoost',output="predicoes_physalis_xgboost.csv",planta="Physalis peruviana")
fim = datetime.now()
print("\n\nFinalizado predição TUBEROSUM:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)
print("\nArquivo salvo:")
print(f"{OUTPUT_DIR}/predicoes_xgboost.csv")


Predição para PHYSALIS




Inciando predição PHYSALIS:  11:54:44

TOP CANDIDATOS com probabilidade de >= 
0.85
pipelineAG/physalis_transcriptoma.fasta, MODELO: XGBoost
                      id                                          descricao  \
10249  IABH01023929.1.p1  IABH01023929.1.p1 GENE.IABH01023929.1~~IABH010...   
7797   IABH01019816.1.p1  IABH01019816.1.p1 GENE.IABH01019816.1~~IABH010...   
9609   IABH01022911.1.p1  IABH01022911.1.p1 GENE.IABH01022911.1~~IABH010...   
4194   IABH01013451.1.p1  IABH01013451.1.p1 GENE.IABH01013451.1~~IABH010...   
9951   IABH01023453.1.p1  IABH01023453.1.p1 GENE.IABH01023453.1~~IABH010...   
...                  ...                                                ...   
9815   IABH01023231.1.p1  IABH01023231.1.p1 GENE.IABH01023231.1~~IABH010...   
6868   IABH01018357.1.p1  IABH01018357.1.p1 GENE.IABH01018357.1~~IABH010...   
1415   IABH01005251.1.p1  IABH01005251.1.p1 GENE.IABH01005251.1~~IABH010...   
10277  IABH01024037.1.p1  IABH01024037.1.

In [25]:
print(df_validacao)

[{'P_LOOP': 231, 'KINASE2': 31, 'GLPL': 222, 'MHD': 219, 'RNBS_A': 0, 'RNBS_B': 243, 'RNBS_C': 297, 'RNBS_D': 303, 'HRD': 13, 'DFG': 66, 'LRR1': 68, 'LRR2': 130, 'planta': 'Arabidopsis thaliana', 'modelo': 'XGBoost'}, {'P_LOOP': 46, 'KINASE2': 14, 'GLPL': 60, 'MHD': 41, 'RNBS_A': 0, 'RNBS_B': 54, 'RNBS_C': 80, 'RNBS_D': 82, 'HRD': 3, 'DFG': 22, 'LRR1': 7, 'LRR2': 21, 'planta': 'Solanum melongena', 'modelo': 'XGBoost'}, {'P_LOOP': 335, 'KINASE2': 180, 'GLPL': 376, 'MHD': 375, 'RNBS_A': 0, 'RNBS_B': 350, 'RNBS_C': 437, 'RNBS_D': 487, 'HRD': 19, 'DFG': 86, 'LRR1': 100, 'LRR2': 190, 'planta': 'Solanum tuberosum', 'modelo': 'XGBoost'}, {'P_LOOP': 94, 'KINASE2': 31, 'GLPL': 114, 'MHD': 115, 'RNBS_A': 0, 'RNBS_B': 116, 'RNBS_C': 149, 'RNBS_D': 161, 'HRD': 7, 'DFG': 10, 'LRR1': 11, 'LRR2': 48, 'planta': 'Physalis peruviana', 'modelo': 'XGBoost'}]


In [26]:
'''
Aplicação - Modelo Random Forest
'''
print("\n\n Predição Modelo RF")
print("\nPredição para ARABIDOPSIS\n")
inicio = datetime.now()
print("\n\n\nInciando predição ARABIDOPSIS: ", inicio.strftime("%H:%M:%S"))
predizer_transcritos(transcriptoma_arabidopsis_fasta, rf_model,"RF",output="predicoes_arabidopsis_rf.csv",planta="Arabidopsis thaliana")
fim = datetime.now()
print("\n\nFinalizado predição ARABIDOPSIS:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)



 Predição Modelo RF

Predição para ARABIDOPSIS




Inciando predição ARABIDOPSIS:  11:56:46

TOP CANDIDATOS com probabilidade de >= 
0.85
pipelineAG/arabidopsis_transcriptoma.fasta, MODELO: RF
                     id                                          descricao  \
9546     NM_124542.3.p1  NM_124542.3.p1 GENE.NM_124542.3~~NM_124542.3.p...   
4720  NM_001344946.1.p1  NM_001344946.1.p1 GENE.NM_001344946.1~~NM_0013...   
4723  NM_001344948.1.p1  NM_001344948.1.p1 GENE.NM_001344948.1~~NM_0013...   
4375  NM_001344503.1.p1  NM_001344503.1.p1 GENE.NM_001344503.1~~NM_0013...   
9182     NM_123739.3.p1  NM_123739.3.p1 GENE.NM_123739.3~~NM_123739.3.p...   
...                 ...                                                ...   
1786  NM_001341144.1.p1  NM_001341144.1.p1 GENE.NM_001341144.1~~NM_0013...   
9433     NM_124291.2.p1  NM_124291.2.p1 GENE.NM_124291.2~~NM_124291.2.p...   
8490     NM_121841.2.p1  NM_121841.2.p1 GENE.NM_121841.2~~NM_121841.2.p...   
8473     NM_121802.2.p1  

In [27]:
print("\nPredição para MELONGENA\n")
inicio = datetime.now()
print("\n\n\nInciando predição MELONGENA: ", inicio.strftime("%H:%M:%S"))
predizer_transcritos(transcriptoma_melongena_fasta, rf_model,'RF',output="predicoes_melongena_rf.csv",planta="Solanum melongena")
fim = datetime.now()
print("\n\nFinalizado predição MELONGENA:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)


Predição para MELONGENA




Inciando predição MELONGENA:  12:10:28

TOP CANDIDATOS com probabilidade de >= 
0.85
pipelineAG/melongena_transcriptoma.fasta, MODELO: RF
                     id                                          descricao  \
3221  GBEF01023314.1.p1  GBEF01023314.1.p1 GENE.GBEF01023314.1~~GBEF010...   
3220  GBEF01023313.1.p1  GBEF01023313.1.p1 GENE.GBEF01023313.1~~GBEF010...   
9238  GBEF01044458.1.p1  GBEF01044458.1.p1 GENE.GBEF01044458.1~~GBEF010...   
4489  GBEF01026077.1.p1  GBEF01026077.1.p1 GENE.GBEF01026077.1~~GBEF010...   
4488  GBEF01026076.1.p1  GBEF01026076.1.p1 GENE.GBEF01026076.1~~GBEF010...   
4337  GBEF01025817.1.p1  GBEF01025817.1.p1 GENE.GBEF01025817.1~~GBEF010...   
4338  GBEF01025818.1.p1  GBEF01025818.1.p1 GENE.GBEF01025818.1~~GBEF010...   
3222  GBEF01023315.1.p1  GBEF01023315.1.p1 GENE.GBEF01023315.1~~GBEF010...   
3223  GBEF01023316.1.p1  GBEF01023316.1.p1 GENE.GBEF01023316.1~~GBEF010...   
274   GBEF01001042.1.p1  GBEF01001042.1.p1 GENE.GBEF0

In [28]:
print("\nPredição para TUBEROSUM\n")
inicio = datetime.now()
print("\n\n\nInciando predição TUBEROSUM: ", inicio.strftime("%H:%M:%S"))
predizer_transcritos(transcriptoma_tuberosum_fasta, rf_model,"RF",output="predicoes_tuberosum_rf.csv",planta="Solanum tuberosum")
fim = datetime.now()
print("\n\nFinalizado predição TUBEROSUM:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)




Predição para TUBEROSUM




Inciando predição TUBEROSUM:  12:22:38

TOP CANDIDATOS com probabilidade de >= 
0.85
pipelineAG/tuberosum_transcriptoma.fasta, MODELO: RF
                      id                                          descricao  \
1960   XM_006356330.2.p1  XM_006356330.2.p1 GENE.XM_006356330.2~~XM_0063...   
1986   XM_006356384.2.p1  XM_006356384.2.p1 GENE.XM_006356384.2~~XM_0063...   
7736   XM_015304639.1.p1  XM_015304639.1.p1 GENE.XM_015304639.1~~XM_0153...   
8918   XM_015312157.1.p1  XM_015312157.1.p1 GENE.XM_015312157.1~~XM_0153...   
8919   XM_015312158.1.p1  XM_015312158.1.p1 GENE.XM_015312158.1~~XM_0153...   
...                  ...                                                ...   
6124   XM_006366011.2.p1  XM_006366011.2.p1 GENE.XM_006366011.2~~XM_0063...   
6125   XM_006366012.2.p1  XM_006366012.2.p1 GENE.XM_006366012.2~~XM_0063...   
2869   XM_006358404.2.p1  XM_006358404.2.p1 GENE.XM_006358404.2~~XM_0063...   
10092  XM_015314339.1.p1  XM_015314339.1.p1

In [29]:
print("\nPredição para PHYSALIS\n")
inicio = datetime.now()
print("\n\n\nInciando predição PHYSALIS: ", inicio.strftime("%H:%M:%S"))
predizer_transcritos(transcriptoma_physalis_fasta, rf_model,"RF",output="predicoes_physalis_rf.csv",planta="Physalis peruviana")
fim = datetime.now()
print("\n\nFinalizado predição PHYSALIS:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)

print("\nArquivo salvo:")
print(f"{OUTPUT_DIR}/predicoes_rf.csv")



Predição para PHYSALIS




Inciando predição PHYSALIS:  12:36:29

TOP CANDIDATOS com probabilidade de >= 
0.85
pipelineAG/physalis_transcriptoma.fasta, MODELO: RF
                      id                                          descricao  \
10242  IABH01023913.1.p1  IABH01023913.1.p1 GENE.IABH01023913.1~~IABH010...   
10241  IABH01023911.1.p1  IABH01023911.1.p1 GENE.IABH01023911.1~~IABH010...   
10240  IABH01023909.1.p1  IABH01023909.1.p1 GENE.IABH01023909.1~~IABH010...   
10938  IABH01029912.1.p1  IABH01029912.1.p1 GENE.IABH01029912.1~~IABH010...   
711    IABH01002505.1.p1  IABH01002505.1.p1 GENE.IABH01002505.1~~IABH010...   
710    IABH01002504.1.p1  IABH01002504.1.p1 GENE.IABH01002504.1~~IABH010...   
4194   IABH01013451.1.p1  IABH01013451.1.p1 GENE.IABH01013451.1~~IABH010...   
9609   IABH01022911.1.p1  IABH01022911.1.p1 GENE.IABH01022911.1~~IABH010...   
10220  IABH01023861.1.p1  IABH01023861.1.p1 GENE.IABH01023861.1~~IABH010...   
7797   IABH01019816.1.p1  IABH01019816.1.p1 GE

In [30]:
'''
Aplicação - Modelo SVM
'''
print("\n\n Predição Modelo SVM")
print("\nPredição para ARABIDOPSIS\n")
inicio = datetime.now()
print("\n\n\nInciando predição ARABIDOPSIS: ", inicio.strftime("%H:%M:%S"))
predizer_transcritos(transcriptoma_arabidopsis_fasta, svm_model,"SVM",output="predicoes_arabidopsis_svm.csv",planta="Arabidopsis thaliana")
fim = datetime.now()
print("\n\nFinalizado predição ARABIDOPSIS:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)



 Predição Modelo SVM

Predição para ARABIDOPSIS




Inciando predição ARABIDOPSIS:  12:51:17

TOP CANDIDATOS com probabilidade de >= 
0.85
pipelineAG/arabidopsis_transcriptoma.fasta, MODELO: SVM
                     id                                          descricao  \
9348     NM_124096.4.p1  NM_124096.4.p1 GENE.NM_124096.4~~NM_124096.4.p...   
3611  NM_001343517.1.p1  NM_001343517.1.p1 GENE.NM_001343517.1~~NM_0013...   
4374  NM_001344502.1.p1  NM_001344502.1.p1 GENE.NM_001344502.1~~NM_0013...   
4376  NM_001344504.1.p1  NM_001344504.1.p1 GENE.NM_001344504.1~~NM_0013...   
4373  NM_001344501.1.p1  NM_001344501.1.p1 GENE.NM_001344501.1~~NM_0013...   
...                 ...                                                ...   
4378  NM_001344506.1.p1  NM_001344506.1.p1 GENE.NM_001344506.1~~NM_0013...   
4377  NM_001344505.1.p1  NM_001344505.1.p1 GENE.NM_001344505.1~~NM_0013...   
4405  NM_001344527.1.p1  NM_001344527.1.p1 GENE.NM_001344527.1~~NM_0013...   
4511  NM_001344665.1.p1

In [31]:
print("\nPredição para MELONGENA\n")
inicio = datetime.now()
print("\n\n\nInciando predição MELONGENA: ", inicio.strftime("%H:%M:%S"))
predizer_transcritos(transcriptoma_melongena_fasta, svm_model,'SVM',output="predicoes_melongena_svm.csv",planta="Solanum melongena")
fim = datetime.now()
print("\n\nFinalizado predição MELONGENA:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)


Predição para MELONGENA




Inciando predição MELONGENA:  12:51:22

TOP CANDIDATOS com probabilidade de >= 
0.85
pipelineAG/melongena_transcriptoma.fasta, MODELO: SVM
                     id                                          descricao  \
4488  GBEF01026076.1.p1  GBEF01026076.1.p1 GENE.GBEF01026076.1~~GBEF010...   
3221  GBEF01023314.1.p1  GBEF01023314.1.p1 GENE.GBEF01023314.1~~GBEF010...   
4489  GBEF01026077.1.p1  GBEF01026077.1.p1 GENE.GBEF01026077.1~~GBEF010...   
3223  GBEF01023316.1.p1  GBEF01023316.1.p1 GENE.GBEF01023316.1~~GBEF010...   
789   GBEF01005143.1.p1  GBEF01005143.1.p1 GENE.GBEF01005143.1~~GBEF010...   
3220  GBEF01023313.1.p1  GBEF01023313.1.p1 GENE.GBEF01023313.1~~GBEF010...   
9160  GBEF01044249.1.p1  GBEF01044249.1.p1 GENE.GBEF01044249.1~~GBEF010...   
274   GBEF01001042.1.p1  GBEF01001042.1.p1 GENE.GBEF01001042.1~~GBEF010...   
9162  GBEF01044251.1.p1  GBEF01044251.1.p1 GENE.GBEF01044251.1~~GBEF010...   
3222  GBEF01023315.1.p1  GBEF01023315.1.p1 GENE.GBEF

In [32]:
print("\nPredição para TUBEROSUM\n")
inicio = datetime.now()
print("\n\n\nInciando predição TUBEROSUM: ", inicio.strftime("%H:%M:%S"))
predizer_transcritos(transcriptoma_tuberosum_fasta, svm_model,"SVM",output="predicoes_tuberosum_svm.csv",planta="Solanum tuberosum")
fim = datetime.now()
print("\n\nFinalizado predição TUBEROSUM:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)


Predição para TUBEROSUM




Inciando predição TUBEROSUM:  12:51:26

TOP CANDIDATOS com probabilidade de >= 
0.85
pipelineAG/tuberosum_transcriptoma.fasta, MODELO: SVM
                     id                                          descricao  \
9242  XM_015312842.1.p1  XM_015312842.1.p1 GENE.XM_015312842.1~~XM_0153...   
7736  XM_015304639.1.p1  XM_015304639.1.p1 GENE.XM_015304639.1~~XM_0153...   
7737  XM_015304640.1.p1  XM_015304640.1.p1 GENE.XM_015304640.1~~XM_0153...   
7738  XM_015304641.1.p1  XM_015304641.1.p1 GENE.XM_015304641.1~~XM_0153...   
7739  XM_015304642.1.p1  XM_015304642.1.p1 GENE.XM_015304642.1~~XM_0153...   
...                 ...                                                ...   
7685  XM_015304550.1.p2  XM_015304550.1.p2 GENE.XM_015304550.1~~XM_0153...   
7687  XM_015304551.1.p1  XM_015304551.1.p1 GENE.XM_015304551.1~~XM_0153...   
7599  XM_015304365.1.p1  XM_015304365.1.p1 GENE.XM_015304365.1~~XM_0153...   
2462  XM_006357493.2.p1  XM_006357493.2.p1 GENE.XM_0

In [33]:
print("\nPredição para PHYSALIS\n")
inicio = datetime.now()
print("\n\n\nInciando predição PHYSALIS: ", inicio.strftime("%H:%M:%S"))
predizer_transcritos(transcriptoma_physalis_fasta, svm_model,"SVM",output="predicoes_physalis_svm.csv",planta="Physalis peruviana")  
fim = datetime.now()
print("\n\nFinalizado predição PHYSALIS:", fim.strftime("%H:%M:%S"))
print("\nTempo decorrido:", fim - inicio)

print("\nArquivo salvo:")
print(f"{OUTPUT_DIR}/predicoes_svm.csv")



Predição para PHYSALIS




Inciando predição PHYSALIS:  12:51:31

TOP CANDIDATOS com probabilidade de >= 
0.85
pipelineAG/physalis_transcriptoma.fasta, MODELO: SVM
                      id                                          descricao  \
10249  IABH01023929.1.p1  IABH01023929.1.p1 GENE.IABH01023929.1~~IABH010...   
9950   IABH01023452.1.p1  IABH01023452.1.p1 GENE.IABH01023452.1~~IABH010...   
10240  IABH01023909.1.p1  IABH01023909.1.p1 GENE.IABH01023909.1~~IABH010...   
9618   IABH01022921.1.p1  IABH01022921.1.p1 GENE.IABH01022921.1~~IABH010...   
9951   IABH01023453.1.p1  IABH01023453.1.p1 GENE.IABH01023453.1~~IABH010...   
...                  ...                                                ...   
9585   IABH01022880.1.p1  IABH01022880.1.p1 GENE.IABH01022880.1~~IABH010...   
7883   IABH01019924.1.p1  IABH01019924.1.p1 GENE.IABH01019924.1~~IABH010...   
7885   IABH01019929.1.p1  IABH01019929.1.p1 GENE.IABH01019929.1~~IABH010...   
1130   IABH01004440.1.p1  IABH01004440.1.p1 G

In [34]:
print(df_validacao)

[{'P_LOOP': 231, 'KINASE2': 31, 'GLPL': 222, 'MHD': 219, 'RNBS_A': 0, 'RNBS_B': 243, 'RNBS_C': 297, 'RNBS_D': 303, 'HRD': 13, 'DFG': 66, 'LRR1': 68, 'LRR2': 130, 'planta': 'Arabidopsis thaliana', 'modelo': 'XGBoost'}, {'P_LOOP': 46, 'KINASE2': 14, 'GLPL': 60, 'MHD': 41, 'RNBS_A': 0, 'RNBS_B': 54, 'RNBS_C': 80, 'RNBS_D': 82, 'HRD': 3, 'DFG': 22, 'LRR1': 7, 'LRR2': 21, 'planta': 'Solanum melongena', 'modelo': 'XGBoost'}, {'P_LOOP': 335, 'KINASE2': 180, 'GLPL': 376, 'MHD': 375, 'RNBS_A': 0, 'RNBS_B': 350, 'RNBS_C': 437, 'RNBS_D': 487, 'HRD': 19, 'DFG': 86, 'LRR1': 100, 'LRR2': 190, 'planta': 'Solanum tuberosum', 'modelo': 'XGBoost'}, {'P_LOOP': 94, 'KINASE2': 31, 'GLPL': 114, 'MHD': 115, 'RNBS_A': 0, 'RNBS_B': 116, 'RNBS_C': 149, 'RNBS_D': 161, 'HRD': 7, 'DFG': 10, 'LRR1': 11, 'LRR2': 48, 'planta': 'Physalis peruviana', 'modelo': 'XGBoost'}, {'P_LOOP': 143, 'KINASE2': 27, 'GLPL': 140, 'MHD': 126, 'RNBS_A': 0, 'RNBS_B': 134, 'RNBS_C': 142, 'RNBS_D': 143, 'HRD': 7, 'DFG': 38, 'LRR1': 33, 'L

In [35]:
df_copia = df_validacao.copy()
df_inicial = pd.DataFrame(df_validacao)

df_validacao = df_inicial.melt(
    id_vars=["planta", "modelo"], 
    var_name="motivo", 
    value_name="contagem"
)
print(df_validacao.head())

df_validacao.to_csv(f"{OUTPUT_DIR}/df_validacao.csv", index=False)

                 planta   modelo  motivo  contagem
0  Arabidopsis thaliana  XGBoost  P_LOOP       231
1     Solanum melongena  XGBoost  P_LOOP        46
2     Solanum tuberosum  XGBoost  P_LOOP       335
3    Physalis peruviana  XGBoost  P_LOOP        94
4  Arabidopsis thaliana       RF  P_LOOP       143


In [36]:
#Scatterplot
plt.figure(figsize=(12,6))
sns.scatterplot(
    data=df_validacao,
    x="motivo",
    y="contagem",
    hue="planta",
    style="modelo",
    s=200
)
# plt.title( "Validação biológica dos motivos conservados")
plt.ylabel("Número de ocorrências")
plt.xlabel("Motivo")
plt.legend(
    bbox_to_anchor=(1.05,1),
    loc="upper left"
)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/Validacao_biologica_dos_motivos_conservados.png")
plt.close()
# plt.show()

#Bubble Chart
plt.figure(figsize=(12,6))
sns.scatterplot(
    data=df_validacao,
    x="motivo",
    y="modelo",
    hue="planta",
    size="contagem",
    sizes=(50,600)
)
# plt.title("Distribuição dos motivos por modelo e espécie")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/distribuicao_dos_motivos_por_modelo_especie.png")
plt.close()
# plt.show()


#Facetas por planta
g = sns.catplot(
    data=df_validacao,
    x="motivo",
    y="contagem",
    hue="modelo",
    col="planta",
    kind="bar",
    height=5,
    aspect=1.2
)
g.set_xticklabels(rotation=45)
plt.savefig(f"{OUTPUT_DIR}/facetas_por_planta.png")
plt.close()
# plt.show()

#por planta

#Physalis peruviana
df_physalis = df_validacao[df_validacao["planta"] == "Physalis peruviana"]
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")
sns.lineplot(
    data=df_physalis,
    x="motivo",
    y="contagem",
    hue="modelo",
    style="modelo",
    markers=True,     # Ativa os marcadores nos pontos
    dashes=False,     # Mantém as linhas contínuas
    linewidth=2,      # Espessura da linha
    markersize=10     # Tamanho dos pontos de dispersão
)

# plt.title("Comparação dos Modelos para Physalis peruviana", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Motivos Conservados", fontsize=12, labelpad=10)
plt.ylabel("Número de Ocorrências (Contagem)", fontsize=12, labelpad=10)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Modelos", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/comparacao_modelos_physalis.png", dpi=300)
plt.close()

#Solanum melongena
df_melongena = df_validacao[df_validacao["planta"] == "Solanum melongena"]
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")
sns.lineplot(
    data=df_melongena,
    x="motivo",
    y="contagem",
    hue="modelo",
    style="modelo",
    markers=True,     # Ativa os marcadores nos pontos
    dashes=False,     # Mantém as linhas contínuas
    linewidth=2,      # Espessura da linha
    markersize=10     # Tamanho dos pontos de dispersão
)

# plt.title("Comparação dos Modelos para Solanum melongena", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Motivos Conservados", fontsize=12, labelpad=10)
plt.ylabel("Número de Ocorrências (Contagem)", fontsize=12, labelpad=10)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Modelos", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/comparacao_modelos_melongena.png", dpi=300)
plt.close()

#Solanum tuberosum
df_tuberosum = df_validacao[df_validacao["planta"] == "Solanum tuberosum"]
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")
sns.lineplot(
    data=df_tuberosum,
    x="motivo",
    y="contagem",
    hue="modelo",
    style="modelo",
    markers=True,     # Ativa os marcadores nos pontos
    dashes=False,     # Mantém as linhas contínuas
    linewidth=2,      # Espessura da linha
    markersize=10     # Tamanho dos pontos de dispersão
)

# plt.title("Comparação dos Modelos para Solanum tuberosum", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Motivos Conservados", fontsize=12, labelpad=10)
plt.ylabel("Número de Ocorrências (Contagem)", fontsize=12, labelpad=10)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Modelos", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/comparacao_modelos_tuberosum.png", dpi=300)
plt.close()

#Arabidopsis thaliana
df_arabidopsis = df_validacao[df_validacao["planta"] == "Arabidopsis thaliana"]
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")
sns.lineplot(
    data=df_arabidopsis,
    x="motivo",
    y="contagem",
    hue="modelo",
    style="modelo",
    markers=True,     # Ativa os marcadores nos pontos
    dashes=False,     # Mantém as linhas contínuas
    linewidth=2,      # Espessura da linha
    markersize=10     # Tamanho dos pontos de dispersão
)

# plt.title("Comparação dos Modelos para Arabidopsis thaliana", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Motivos Conservados", fontsize=12, labelpad=10)
plt.ylabel("Número de Ocorrências (Contagem)", fontsize=12, labelpad=10)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Modelos", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/comparacao_modelos_arabidopsis.png", dpi=300)
plt.close()

#Heatmap
df_validacao["grupo"] = (
    df_validacao["planta"]
    + "_"
    + df_validacao["modelo"]
)

pivot = df_validacao.pivot_table(
    index="motivo",
    columns="grupo",
    values="contagem",
    fill_value=0
)


plt.figure(figsize=(14, 8))
sns.heatmap(
    pivot,
    annot=True,
    fmt="g",
    cmap="YlGnBu",
    linewidths=0.5,
    cbar_kws={'label': 'Contagem'} # Nomeia a barra de cores lateral
)

# plt.title("Distribuição dos Motivos Conservados por Planta e Modelo", fontsize=14, fontweight="bold", pad=15)
plt.ylabel("Motivo", fontsize=12)
plt.xlabel("Grupo (Planta & Modelo)", fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/heatmap_distribuicao_dos_motivos_conservados.png",dpi=300)
plt.close()
# plt.show()


print("========= Fim do Pipeline ============")

========= Fim do Pipeline ============
